## Data Loading and Preview
Load both `Exported.xlsx` and `Import.xlsx` datasets and check their dimensions and initial rows.

In [ ]:
import pandas as pd
from IPython.display import display

# Load the datasets
df_exported = pd.read_excel(r'h:\Compare\Exported.xlsx')
df_import = pd.read_excel(r'h:\Compare\Import.xlsx')

print(f"Exported.xlsx shape: {df_exported.shape}")
print(f"Import.xlsx shape: {df_import.shape}")

In [ ]:
# Preview Exported data
df_exported.head()

In [ ]:
# Preview Import data
df_import.head()

## Comparison
Here we compare the two files based on `SKU`.

In [ ]:
# Fix column name with BOM in Exported.xlsx
if '\ufeffID' in df_exported.columns:
    df_exported = df_exported.rename(columns={'\ufeffID': 'ID'})

# Ensure SKU is string to avoid type mismatches (and fill NaNs)
df_exported['SKU'] = df_exported['SKU'].fillna('MISSING').astype(str)
df_import['SKU'] = df_import['SKU'].fillna('MISSING').astype(str)

skus_exported = set(df_exported['SKU'].unique())
skus_import = set(df_import['SKU'].unique())

print(f"Unique SKUs in Exported: {len(skus_exported)}")
print(f"Unique SKUs in Import: {len(skus_import)}")

missing_in_import = skus_exported - skus_import
missing_in_exported = skus_import - skus_exported

print(f"SKUs in Exported but NOT in Import: {len(missing_in_import)}")
print(f"SKUs in Import but NOT in Exported: {len(missing_in_exported)}")

In [ ]:
# Show details of SKUs missing in Import
df_missing_in_import = df_exported[df_exported['SKU'].isin(missing_in_import)]
df_missing_in_import[['ID', 'SKU', 'Name']].head()

In [ ]:
# Show details of SKUs missing in Exported
df_missing_in_exported = df_import[df_import['SKU'].isin(missing_in_exported)]
df_missing_in_exported[['ID', 'SKU', 'Name']].head()

In [ ]:
# Merge to compare specific columns for matching SKUs
df_merged = pd.merge(df_exported, df_import, on='SKU', suffixes=('_export', '_import'))
print(f"Matching SKUs found: {len(df_merged)}")

# Example: Compare stock status
if 'In stock?_export' in df_merged.columns and 'In stock?_import' in df_merged.columns:
    stock_diff = df_merged[df_merged['In stock?_export'] != df_merged['In stock?_import']]
    print(f"Products with stock status differences: {len(stock_diff)}")
    display(stock_diff[['SKU', 'Name_export', 'In stock?_export', 'In stock?_import']])
elif 'In stock?_export' not in df_merged.columns:
    print("Column 'In stock?' not found in Exported.xlsx")
elif 'In stock?_import' not in df_merged.columns:
    print("Column 'In stock?' not found in Import.xlsx")